# 🔍 Customer Churn Analysis

**Author:** Charan Pasupuleti | **Tools:** Python · Pandas · NumPy · Matplotlib

**Goal:** Identify key drivers of customer churn from a 1,000-row dataset.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
plt.rcParams["figure.facecolor"] = "#0a0a0f"
plt.rcParams["axes.facecolor"]   = "#111118"
plt.rcParams["text.color"]       = "#e8e8f0"
plt.rcParams["axes.labelcolor"]  = "#e8e8f0"
plt.rcParams["xtick.color"]      = "#6b6b88"
plt.rcParams["ytick.color"]      = "#6b6b88"
plt.rcParams["grid.color"]       = "#1e1e2e"
print("Libraries loaded")

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv("../data/customer_churn.csv")
print(f"Shape: {df.shape}")
print(f"Churn rate: {df.churned.mean():.1%}")
df.head()

In [ ]:
print(df.dtypes)
print("
Missing values:")
print(df.isnull().sum())

## 2. Feature Engineering

In [ ]:
df["join_date"] = pd.to_datetime(df["join_date"])
df["revenue_per_login"] = df["monthly_charges"] / df["monthly_logins"].replace(0, np.nan)
df["support_intensity"] = df["support_calls_6m"] / df["tenure_months"].replace(0, 1)
df["is_low_engagement"] = (df["monthly_logins"] < 3).astype(int)
df["is_high_support"]   = (df["support_calls_6m"] > 8).astype(int)
print("Features engineered")
df[["revenue_per_login","support_intensity","is_low_engagement","is_high_support"]].describe()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ["#00e5ff","#a8ff78","#ff6b6b","#ffd166","#c084fc"]

churn_seg = df.groupby("age_segment")["churned"].mean().sort_values(ascending=False) * 100
churn_seg.plot(kind="bar", ax=axes[0], color=colors, edgecolor="none")
axes[0].set_title("Churn by Age Segment", color="#e8e8f0")
axes[0].set_ylabel("Churn Rate (%)")
axes[0].axhline(df["churned"].mean()*100, color="#ff6b6b", linestyle="--", lw=1, label="avg")
axes[0].legend(fontsize=8)
axes[0].tick_params(axis="x", rotation=30)

churn_plan = df.groupby("plan_type")["churned"].mean().sort_values(ascending=False) * 100
churn_plan.plot(kind="bar", ax=axes[1], color=["#ff6b6b","#ffd166","#a8ff78"], edgecolor="none")
axes[1].set_title("Churn by Plan Type", color="#e8e8f0")
axes[1].set_ylabel("Churn Rate (%)")
axes[1].tick_params(axis="x", rotation=0)

churn_region = df.groupby("region")["churned"].mean().sort_values(ascending=False) * 100
churn_region.plot(kind="barh", ax=axes[2], color="#00e5ff", edgecolor="none")
axes[2].set_title("Churn by Region", color="#e8e8f0")
axes[2].set_xlabel("Churn Rate (%)")

plt.tight_layout()
plt.savefig("../outputs/churn_by_segment.png", dpi=150, bbox_inches="tight", facecolor="#0a0a0f")
plt.show()
print("Chart saved")

In [ ]:
numeric_cols = ["tenure_months","monthly_charges","support_calls_6m","monthly_logins","satisfaction_score","churned"]
corr = df[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="RdYlGn", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(numeric_cols, fontsize=8)
ax.set_title("Feature Correlation Matrix", color="#e8e8f0", fontsize=12)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=7, color="black")
plt.tight_layout()
plt.savefig("../outputs/correlation_matrix.png", dpi=150, bbox_inches="tight", facecolor="#0a0a0f")
plt.show()

## 4. Key Findings

In [ ]:
print("=" * 50)
print("  CHURN ANALYSIS — KEY FINDINGS")
print("=" * 50)
overall = df["churned"].mean()
print(f"
Overall churn rate: {overall:.1%}")
print("
Highest risk segments:")
for seg, rate in df.groupby("age_segment")["churned"].mean().sort_values(ascending=False).head(3).items():
    print(f"  {seg}: {rate:.1%}")
print("
Support calls impact:")
print(f"  <=3 calls: {df[df.support_calls_6m<=3].churned.mean():.1%}")
print(f"  >8 calls:  {df[df.support_calls_6m>8].churned.mean():.1%}")
print("
Login frequency impact:")
print(f"  <3 logins/month: {df[df.monthly_logins<3].churned.mean():.1%}")
print(f"  15+ logins/month: {df[df.monthly_logins>=15].churned.mean():.1%}")

---
## 5. Next Steps
- Build logistic regression to score individual churn risk
- Deploy risk scores to Power BI for the CRM team
- A/B test retention offer for Starter + 18-25 segment

*Analysis by Charan Pasupuleti — nagapasupuleti98@outlook.com*